# Ноутбук 04. Бінарна класифікація шаблонів — навчання $f_B$

Цей ноутбук тренує бінарний детектор $f_B: \mathbb{R}^{768} \to \{0, 1\}$ безпосередньо на парах $(\beta(t), \ell(t))$, де $\beta(t)$ — BERT-вектор шаблону, а $\ell(t)$ — фінальна (з можливим експертним коригуванням) бінарна мітка з ноутбука 03. KMeans-кластери **не використовуються** як навчальний сигнал (амендмент архітектури — див. розділи 2.1, 2.3 тези).

Етапи:

1. Завантаження BERT-векторів `(77, 768)` та фінальних міток $\ell(t)$.
2. L2-нормалізація рядків (BERT анізотропний, лінійні класифікатори чутливі до масштабу).
3. Розбиття на train/test 80 % / 20 % зі стратифікацією за $\ell$ на рівні **шаблонів** (Перевірка 1 тези).
4. Навчання GaussianNB / LogisticRegression / LinearSVC з `sample_weight = support(t)` для компенсації дисбалансу.
5. Метрики: **MCC** (основна), F1-macro, ROC-AUC, час навчання та інференсу на запис (CPU, один потік).
6. Baseline: `DummyClassifier(strategy='most_frequent')` — нижня межа адекватності з тези 2.4.
7. Перевірка 2 (теза 2.3): MCC того самого передбачення проти rule-based міток $\ell^{(0)}$ — без переnaвчання.

Вихідний артефакт: `data/processed/classification/metrics_fb.csv`.

In [1]:
from __future__ import annotations

import os

# CPU single-thread for deterministic, reproducible timing. Must be set BEFORE
# numpy/sklearn import — BLAS reads these at C-extension load time.
for _var in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "BLIS_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ[_var] = "1"

import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_PROCESSED = ROOT / 'data' / 'processed'
EMBEDDINGS_DIR = DATA_PROCESSED / 'embeddings'
CLUSTERS_DIR = DATA_PROCESSED / 'clusters'
RESULTS_DIR = DATA_PROCESSED / 'classification'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

from src.classification.trainer import (
    evaluate,
    fit_with_timing,
    inference_seconds_per_record,
    make_baseline,
    make_classifiers,
)
from src.io.persistence import load_json, load_numpy

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f'numpy = {np.__version__}')
print(f'pandas = {pd.__version__}')

numpy = 2.4.4
pandas = 3.0.2


## 1. Завантаження ембедингів, міток та мапінгу

Зчитуємо чотири артефакти попередніх ноутбуків:

* `embeddings/zookeeper_full_embeddings.npy` $(77, 768)$ `float32` — BERT [CLS]-вектори.
* `embeddings/zookeeper_full_id_mapping.json` — порядок `row_index → template_id`.
* `clusters/template_to_binary_label.json` — фінальна мітка $\ell(t)$ + `support(t)` + `level_counts`.
* `clusters/template_to_rule_label.json` — мінімальний контракт $\{\texttt{template\_id}, \texttt{rule\_label}\}$ для Перевірки 2.

Стикуємо їх через `template_id` так, щоб усі чотири масиви — `X`, `y`, `w`, `y_rule` — мали однаковий порядок рядків (тобто порядок з `id_mapping`).

In [2]:
embeddings = load_numpy(EMBEDDINGS_DIR / 'zookeeper_full_embeddings.npy')
id_mapping: list[dict] = load_json(EMBEDDINGS_DIR / 'zookeeper_full_id_mapping.json')
template_to_label = load_json(CLUSTERS_DIR / 'template_to_binary_label.json')
template_to_rule = load_json(CLUSTERS_DIR / 'template_to_rule_label.json')

# Lookup dicts keyed by template_id.
final_by_tid: dict[int, int] = {r['template_id']: r['final_label'] for r in template_to_label}
support_by_tid: dict[int, int] = {r['template_id']: r['support'] for r in template_to_label}
rule_by_tid: dict[int, int] = {r['template_id']: r['rule_label'] for r in template_to_rule}

# Every embedding row must have a label, and vice versa — on both files.
embed_tids = {entry['template_id'] for entry in id_mapping}
assert embed_tids == set(final_by_tid), \
    f'embedding/label template_id mismatch: {embed_tids ^ set(final_by_tid)}'
assert embed_tids == set(rule_by_tid), \
    f'embedding/rule  template_id mismatch: {embed_tids ^ set(rule_by_tid)}'

# Align everything to id_mapping row order.
# X shape (77, 768) float32; y/w/y_rule shape (77,).
X = embeddings.astype(np.float32)
y = np.array([final_by_tid[m['template_id']] for m in id_mapping], dtype=int)
w = np.array([support_by_tid[m['template_id']] for m in id_mapping], dtype=float)
y_rule = np.array([rule_by_tid[m['template_id']] for m in id_mapping], dtype=int)

assert X.shape == (77, 768), X.shape
assert y.shape == (77,) and w.shape == (77,) and y_rule.shape == (77,)
assert set(np.unique(y).tolist()) == {0, 1}, f'y must contain both classes, got {set(y.tolist())}'

print(f'X      = {X.shape} {X.dtype}')
print(f'y      = {y.shape} (binary)')
print(f'w      = {w.shape} (template support)')
print(f'y_rule = {y_rule.shape} (rule-only labels for Перевірка 2)')

X      = (77, 768) float32
y      = (77,) (binary)
w      = (77,) (template support)
y_rule = (77,) (rule-only labels for Перевірка 2)


## 2. L2-нормалізація рядків

Без нормалізації лінійні класифікатори (особливо LinearSVC та LogReg) перекошуються в напрямку шаблонів з більшою нормою BERT-вектора. У ноутбуку 03 §2 ми бачили, що норми лежать у вузькій смузі $[13.4, 15.8]$, але навіть таке коливання впливає на margin/гіперплощину. Тому переводимо всі рядки на одиничну гіперсферу (та сама нормалізація, що в діагностичному KMeans).

In [3]:
X_norm = normalize(X, norm='l2', axis=1).astype(np.float32)
# X_norm shape (77, 768) float32; row norms == 1 ± eps.

post_norms = np.linalg.norm(X_norm, axis=1)
assert np.allclose(post_norms, 1.0, atol=1e-5), 'L2 normalization did not produce unit vectors'

print(f'X_norm = {X_norm.shape} {X_norm.dtype}')
print(f'row-norm min/max = {post_norms.min():.6f} / {post_norms.max():.6f}')

X_norm = (77, 768) float32
row-norm min/max = 1.000000 / 1.000000


## 3. Баланс класів

На рівні **шаблонів** очікувано приблизно $16$ позитивних / $61$ негативний (у поточному стані без експертних корекцій — $18 / 59$). На рівні **записів** ситуація обернена: позитивний клас домінує (~$66\%$ трафіку — переважно WARN-шаблони). Саме тому `sample_weight = support(t)` — навчання зміщується на користь практично домінуючих подій без зміни постановки задачі (теза 2.1).

Тут же друкуємо повну (по всіх 77 шаблонах) кількість розбіжностей між експертною та rule-based розмітками — це базова цифра для Перевірки 2.

In [4]:
class_counts = Counter(y.tolist())
print(f'templates: y=0 -> {class_counts[0]}, y=1 -> {class_counts[1]} (of {y.size})')

# Weighted record-level distribution (sample_weight = support).
pos_records = int(w[y == 1].sum())
neg_records = int(w[y == 0].sum())
total_records = pos_records + neg_records
print(
    f'records (via support): y=0 -> {neg_records} '
    f'({100 * neg_records / total_records:.2f} %), '
    f'y=1 -> {pos_records} ({100 * pos_records / total_records:.2f} %)'
)

# Rule vs expert label disagreement on the FULL template set.
n_disagree_total = int((y != y_rule).sum())
print(f'rule vs expert disagreement (all templates): {n_disagree_total} of {y.size}')

templates: y=0 -> 61, y=1 -> 16 (of 77)
records (via support): y=0 -> 70732 (95.10 %), y=1 -> 3648 (4.90 %)
rule vs expert disagreement (all templates): 16 of 77


## 4. Train/test split на рівні шаблонів (Перевірка 1)

Стратифіковане розбиття $80\,\% / 20\,\%$ на $(X, y, w, y_{\text{rule}})$ з `random_state=42`. Розбиття саме **на рівні шаблонів** (не записів) запобігає витоку даних: різні записи одного шаблону не можуть одночасно потрапити у train та test.

За малої потужності позитивного класу (~$16$) тестова вибірка містить лише ~$3$–$4$ позитиви — друкуємо точні кількості та явно вказуємо caveat у разі $<3$ прикладів класу (теза 2.4).

In [5]:
X_train, X_test, y_train, y_test, w_train, w_test, y_rule_train, y_rule_test = train_test_split(
    X_norm, y, w, y_rule,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f'train: X={X_train.shape}, y={y_train.shape}, balance = {dict(Counter(y_train.tolist()))}')
print(f'test : X={X_test.shape}, y={y_test.shape}, balance = {dict(Counter(y_test.tolist()))}')

test_class_counts = Counter(y_test.tolist())
for cls in (0, 1):
    n_cls = test_class_counts[cls]
    if n_cls < 3:
        print(
            f'⚠ caveat: class {cls} has only {n_cls} sample(s) in the test split; '
            'MCC and ROC-AUC are statistically unstable (теза 2.4).'
        )

n_disagree_test = int((y_test != y_rule_test).sum())
print(f'rule vs expert disagreement (test split): {n_disagree_test} of {y_test.size}')

train: X=(61, 768), y=(61,), balance = {1: 13, 0: 48}
test : X=(16, 768), y=(16,), balance = {0: 13, 1: 3}
rule vs expert disagreement (test split): 3 of 16


## 5. Навчання трьох класифікаторів + baseline

Для кожної моделі:

1. `fit(X_train, y_train, sample_weight=w_train)` — час `fit` записується.
2. `predict(X_test)` → `evaluate(...)` рахує MCC, F1-macro, ROC-AUC та `MCC_vs_rulebased` (Перевірка 2).
3. `inference_seconds_per_record(model, X_test, repeats=200)` — середній час `predict` на один запис.

`DummyClassifier(strategy='most_frequent')` навчається тими ж даними; його MCC має дорівнювати $0$ — це нижня межа адекватності (теза 2.4).

In [6]:
classifiers = make_classifiers(random_state=RANDOM_STATE)
classifiers['Baseline'] = make_baseline(random_state=RANDOM_STATE)

per_model: dict[str, dict[str, float]] = {}

for name, clf in classifiers.items():
    train_s = fit_with_timing(clf, X_train, y_train, sample_weight=w_train)
    metrics = evaluate(clf, X_test, y_test, y_rule_test)
    infer_s = inference_seconds_per_record(clf, X_test, repeats=200)
    metrics['train_s'] = train_s
    metrics['infer_s_per_rec'] = infer_s
    per_model[name] = metrics
    print(
        f'{name:<10}  MCC={metrics["MCC"]:+.4f}  '
        f'F1={metrics["F1_macro"]:.4f}  '
        f'ROC-AUC={metrics["ROC_AUC"]:.4f}  '
        f'MCC(rule)={metrics["MCC_vs_rulebased"]:+.4f}  '
        f'fit={train_s:.4g}s  infer={infer_s:.4g}s/rec'
    )

GaussianNB  MCC=+0.0000  F1=0.4483  ROC-AUC=0.5000  MCC(rule)=+0.0000  fit=0.001259s  infer=3.722e-06s/rec
LogReg      MCC=+0.5375  F1=0.7143  ROC-AUC=0.8462  MCC(rule)=-0.1491  fit=0.007715s  infer=1.61e-06s/rec
LinearSVC   MCC=+0.3026  F1=0.6444  ROC-AUC=0.9231  MCC(rule)=-0.2182  fit=0.03012s  infer=1.605e-06s/rec
Baseline    MCC=+0.0000  F1=0.4483  ROC-AUC=0.5000  MCC(rule)=+0.0000  fit=9.329e-05s  infer=4.816e-06s/rec


In [7]:
# Full-domain Перевірка 2 (теза 2.3): predict the already-trained models on ALL 77
# template vectors and recompute MCC against (a) expert final labels y and (b) rule
# labels y_rule. The trained models are NOT refit — this is a pure prediction pass.
#
# Why a second cross-check exists: with only ~16 test templates and a small number
# of expert corrections, the stratified split routinely places all corrected
# templates in train, leaving y_test == y_rule_test on the test split. The
# existing column `MCC_vs_rulebased` then equals `MCC` by construction and is
# vacuous. The full-domain pair below is the meaningful Перевірка 2 signal; the
# test-only column is kept as an honest record of the small-sample limitation.
n_corrected_total = int((y != y_rule).sum())
n_corrected_in_test = int((y_test != y_rule_test).sum())
print(
    f'expert corrections — total: {n_corrected_total}, in test split: {n_corrected_in_test}'
)
if n_corrected_in_test == 0:
    print(
        '⚠ caveat: MCC_vs_rulebased on the test split is vacuous '
        '(rule == expert there). The MCC_full_expert / MCC_full_rulebased '
        'pair below is the meaningful Перевірка 2 signal (теза 2.3).'
    )

for name, clf in classifiers.items():
    full_metrics = evaluate(clf, X_norm, y, y_rule)
    per_model[name]['MCC_full_expert'] = full_metrics['MCC']
    per_model[name]['MCC_full_rulebased'] = full_metrics['MCC_vs_rulebased']
    per_model[name]['n_corrected_in_test'] = n_corrected_in_test
    print(
        f'{name:<10}  MCC_full_expert={full_metrics["MCC"]:+.4f}  '
        f'MCC_full_rulebased={full_metrics["MCC_vs_rulebased"]:+.4f}'
    )

expert corrections — total: 16, in test split: 3
GaussianNB  MCC_full_expert=+0.3189  MCC_full_rulebased=+0.2956
LogReg      MCC_full_expert=+0.6649  MCC_full_rulebased=+0.3148
LinearSVC   MCC_full_expert=+0.8796  MCC_full_rulebased=+0.2707
Baseline    MCC_full_expert=+0.0000  MCC_full_rulebased=+0.0000


## 6. Підсумкова таблиця та експорт

Збираємо результати у `DataFrame` з фіксованим порядком рядків (`GaussianNB → LogReg → LinearSVC → Baseline`) і колонок:

`MCC, F1_macro, ROC_AUC, MCC_vs_rulebased, MCC_full_expert, MCC_full_rulebased, n_corrected_in_test, train_s, infer_s_per_rec`.

Друк округлюється до $4$ значущих цифр; CSV зберігається з повною точністю.

**Інтерпретація Перевірки 2 (теза 2.3).** Через малу потужність тестового сплету ($n_{\text{test}}=16$) та стратифіковане розбиття всі експертно скориговані шаблони, як правило, потрапляють у train. Колонка `n_corrected_in_test` робить цей факт явним: якщо вона дорівнює $0$, то на тесті `y_rule_test ≡ y_test`, і `MCC_vs_rulebased == MCC` за побудовою — це чесний індикатор обмеження малої вибірки, а не сигнал якості. Тому додано пару **повнодоменних** значень `MCC_full_expert` та `MCC_full_rulebased` — це MCC того ж самого передбачення на **усіх 77 шаблонах**, не переnaвчаючи моделей; саме ця пара є змістовним сигналом Перевірки 2. Висновок про семантичну природу класифікатора робиться лише тоді, коли обидва повнодоменних MCC високі **і** хоча б один шаблон має `rule_label ≠ final_label`.

In [8]:
row_order = ['GaussianNB', 'LogReg', 'LinearSVC', 'Baseline']
col_order = [
    'MCC',
    'F1_macro',
    'ROC_AUC',
    'MCC_vs_rulebased',
    'MCC_full_expert',
    'MCC_full_rulebased',
    'n_corrected_in_test',
    'train_s',
    'infer_s_per_rec',
]

metrics_df = pd.DataFrame.from_dict(per_model, orient='index').loc[row_order, col_order]

# 4 sig figs for printed view; CSV keeps full precision.
with pd.option_context('display.float_format', '{:.4g}'.format):
    print(metrics_df)

csv_path = RESULTS_DIR / 'metrics_fb.csv'
metrics_df.to_csv(csv_path, index_label='model')
print(f'\nsaved: {csv_path}  ({csv_path.stat().st_size} bytes)')

# Inline checks (теза 2.3 Перевірка 1 + sanity floor + full-domain coverage).
assert metrics_df.shape == (4, len(col_order)), metrics_df.shape
assert not metrics_df['MCC'].isna().any(), 'MCC column must not contain NaN'
assert not metrics_df['MCC_full_expert'].isna().any(), (
    'MCC_full_expert column must not contain NaN'
)
assert abs(metrics_df.loc['Baseline', 'MCC']) < 1e-9, (
    f"baseline MCC expected 0 (constant prediction); got {metrics_df.loc['Baseline', 'MCC']}"
)
print('asserts passed.')

              MCC  F1_macro  ROC_AUC  MCC_vs_rulebased  MCC_full_expert  \
GaussianNB      0    0.4483      0.5                 0           0.3189   
LogReg     0.5375    0.7143   0.8462           -0.1491           0.6649   
LinearSVC  0.3026    0.6444   0.9231           -0.2182           0.8796   
Baseline        0    0.4483      0.5                 0                0   

            MCC_full_rulebased  n_corrected_in_test   train_s  infer_s_per_rec  
GaussianNB              0.2956                    3  0.001259        3.722e-06  
LogReg                  0.3148                    3  0.007715         1.61e-06  
LinearSVC               0.2707                    3   0.03012        1.605e-06  
Baseline                     0                    3 9.329e-05        4.816e-06  

saved: /Users/roman/Personal/dyploma/data/processed/classification/metrics_fb.csv  (683 bytes)
asserts passed.


## 7. Повнодоменні матриці плутанини

Розділ 3.5 тези потребує **TP, FP, FN, TN** кожного класифікатора на **всіх 77 шаблонах** (повний домен, без train/test поділу). Це не оцінка узагальнення — це діагностика розподілу помилок, яку розділ обговорює якісно. Також тут перераховуємо $\mathrm{MCC}_{\text{full,expert}}$ і робимо tie-back до значень з `metrics_fb.csv` (точність $10^{-9}$).


In [9]:
from sklearn.metrics import matthews_corrcoef

# Predict each already-trained model on the full 77-template matrix.
# y_pred_full shape (77,); kept in `predictions_full` so §8 can reuse LinearSVC.
predictions_full: dict[str, np.ndarray] = {}
confusion_rows: list[dict] = []

for name, clf in classifiers.items():
    y_pred_full = clf.predict(X_norm).astype(int)
    assert y_pred_full.shape == (77,), y_pred_full.shape
    predictions_full[name] = y_pred_full

    tp = int(((y == 1) & (y_pred_full == 1)).sum())
    fp = int(((y == 0) & (y_pred_full == 1)).sum())
    fn = int(((y == 1) & (y_pred_full == 0)).sum())
    tn = int(((y == 0) & (y_pred_full == 0)).sum())
    mcc_recomputed = float(matthews_corrcoef(y, y_pred_full))

    confusion_rows.append({
        'model': name,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'MCC_full_recomputed': mcc_recomputed,
        'total_errors': fp + fn,
    })

confusion_df = pd.DataFrame(
    confusion_rows,
    columns=['model', 'TP', 'FP', 'FN', 'TN', 'MCC_full_recomputed', 'total_errors'],
)

# Tie-back: each MCC_full_recomputed must match metrics_fb.csv to 1e-9.
# Both use the same matthews_corrcoef call on the same (y, y_pred_full) pair,
# so the assertion guards against accidental signature drift, not numerical noise.
metrics_fb_df = pd.read_csv(csv_path).set_index('model')
for row in confusion_rows:
    expected = float(metrics_fb_df.loc[row['model'], 'MCC_full_expert'])
    assert abs(row['MCC_full_recomputed'] - expected) < 1e-9, (
        f"MCC tie-back failed for {row['model']}: "
        f"recomputed={row['MCC_full_recomputed']} vs metrics_fb={expected}"
    )

with pd.option_context('display.float_format', '{:.4g}'.format):
    print(confusion_df.to_string(index=False))

confusion_path = RESULTS_DIR / 'confusion_full.csv'
confusion_df.to_csv(confusion_path, index=False)
print(f'\nsaved: {confusion_path}  ({confusion_path.stat().st_size} bytes)')

     model  TP  FP  FN  TN  MCC_full_recomputed  total_errors
GaussianNB   2   0  14  61               0.3189            14
    LogReg   8   0   8  61               0.6649             8
 LinearSVC  14   1   2  60               0.8796             3
  Baseline   0   0  16  61                    0            16

saved: /Users/roman/Personal/dyploma/data/processed/classification/confusion_full.csv  (198 bytes)


## 8. Списки помилок LinearSVC на повному домені

LinearSVC — фінальний детектор $f_B$ (найвищий $\mathrm{MCC}_{\text{full,expert}}$ серед трьох). Розділ 3.5 обговорює його **FP** (помилкові тривоги: rule_label/expert = 0, але модель передбачає 1) та **FN** (пропуски: rule_label/expert = 1, але модель передбачає 0) поіменно — за `template_id`. TP/TN зберігаємо як int-списки (мала вага в обговоренні), FP/FN — як повні об'єкти з `support` та текстом шаблону.


In [10]:
from src.io.persistence import save_json

# template_id -> {support, template} lookup from the row-aligned id_mapping.
tid_to_meta: dict[int, dict] = {
    m['template_id']: {'support': int(m['support']), 'template': str(m['template'])}
    for m in id_mapping
}
row_tids: list[int] = [m['template_id'] for m in id_mapping]

y_pred_lin = predictions_full['LinearSVC']

tp_ids: list[int] = sorted(row_tids[i] for i in range(77) if y[i] == 1 and y_pred_lin[i] == 1)
fp_ids: list[int] = sorted(row_tids[i] for i in range(77) if y[i] == 0 and y_pred_lin[i] == 1)
fn_ids: list[int] = sorted(row_tids[i] for i in range(77) if y[i] == 1 and y_pred_lin[i] == 0)
tn_ids: list[int] = sorted(row_tids[i] for i in range(77) if y[i] == 0 and y_pred_lin[i] == 0)

assert len(tp_ids) + len(fp_ids) + len(fn_ids) + len(tn_ids) == 77, \
    'counts must partition 77 templates'


def _row(tid: int) -> dict:
    """Build a {template_id, support, template} object for FP/FN export."""
    meta = tid_to_meta[tid]
    return {'template_id': int(tid), 'support': meta['support'], 'template': meta['template']}


fp_objects: list[dict] = [_row(t) for t in fp_ids]
fn_objects: list[dict] = [_row(t) for t in fn_ids]


def _print_block(label: str, objs: list[dict]) -> None:
    """Pretty-print an FP or FN block with the template truncated to 80 chars."""
    print(f'\n{label} ({len(objs)})')
    if not objs:
        print('  (none)')
        return
    print(f"  {'tid':>4}  {'support':>7}  template")
    for o in objs:
        t = o['template']
        t_short = t if len(t) <= 80 else t[:77] + '...'
        print(f"  {o['template_id']:>4}  {o['support']:>7}  {t_short}")


_print_block('FP (false positives)', fp_objects)
_print_block('FN (false negatives)', fn_objects)

errors_payload = {
    'TP': [int(t) for t in tp_ids],
    'FP': fp_objects,
    'FN': fn_objects,
    'TN': [int(t) for t in tn_ids],
}
errors_path = RESULTS_DIR / 'linearsvc_errors_full.json'
save_json(errors_payload, errors_path)
print(f'\nsaved: {errors_path}  ({errors_path.stat().st_size} bytes)')


FP (false positives) (1)
   tid  support  template
     5       36  Purge task is not scheduled.

FN (false negatives) (2)
   tid  support  template
    62       15  Commiting zxid <HEX> from /<IP> not first!
    63       15  First is <HEX>

saved: /Users/roman/Personal/dyploma/data/processed/classification/linearsvc_errors_full.json  (965 bytes)


## 9. Запис-рівневі статистики фільтра (LinearSVC)

Розділ 3.6 тези заявляє цифру практичного ефекту фільтра — скільки **записів** із 74380 ZooKeeper-логу LinearSVC залишить як інформативні, а скільки відсіє як шум. Метод: проектуємо передбачення моделі з 77 шаблонів на 74380 записів через зіставлення `record_to_template`. Якщо файл `record_to_template.npy` ще не збережено, відтворюємо його одноразово через `DrainParser` з тими самими параметрами, що і в ноутбуках 01/03 (Drain детермінований), і кешуємо у `data/processed/records/` для майбутніх ноутбуків.

Порівнюємо передбачення `pred_per_record` зі справжніми мітками `record_to_binary_label.npy` (з ноутбука 03) — це експертна референція на рівні записів. Звітуємо $n_{\text{інф}}$, $n_{\text{шум}}$, відсотки, та запис-рівневу матрицю плутанини з MCC.


In [11]:
import logging

from src.drain.parser import DrainParser
from src.io.persistence import load_numpy as _load_numpy_local, save_numpy

# Per-line drain logging is noisy and uninteresting here — reparse is identical to nb03 §6.
logging.getLogger('drain3').setLevel(logging.WARNING)
logging.getLogger('src.drain.parser').setLevel(logging.WARNING)

RECORDS_DIR = DATA_PROCESSED / 'records'
RECORDS_DIR.mkdir(parents=True, exist_ok=True)
record_to_template_path = RECORDS_DIR / 'record_to_template.npy'

# record_to_template shape (74380,) int32 — template_id assigned by Drain to each raw log line.
if record_to_template_path.exists():
    record_to_template = _load_numpy_local(record_to_template_path)
    print(f'loaded existing record_to_template: {record_to_template.shape} {record_to_template.dtype}')
else:
    # Same DrainParser config as nb01/nb03 → same partition (Drain is deterministic).
    normalized_path = DATA_PROCESSED / 'zookeeper_full_normalized.txt'
    with normalized_path.open('r', encoding='utf-8') as f:
        normalized_lines = [line.rstrip('\n') for line in f]
    assert len(normalized_lines) == 74380, f'expected 74380 normalized lines, got {len(normalized_lines)}'

    parser = DrainParser(similarity_threshold=0.4, depth=4, max_children=100)
    results = parser.parse_stream(normalized_lines)
    record_to_template = np.array([r.template_id for r in results], dtype=np.int32)
    save_numpy(record_to_template, record_to_template_path)
    print(f'derived & saved record_to_template: {record_to_template.shape} {record_to_template.dtype}')

assert record_to_template.shape == (74380,), record_to_template.shape
assert np.issubdtype(record_to_template.dtype, np.integer), record_to_template.dtype
# Set of template_ids in the per-record array must equal the 77 in id_mapping (no drift).
seen_tids = {int(t) for t in np.unique(record_to_template)}
known_tids = {m['template_id'] for m in id_mapping}
assert seen_tids == known_tids, f'template_id mismatch (∆): {seen_tids ^ known_tids}'

# --- Project LinearSVC predictions from templates to records ---

pred_per_template = predictions_full['LinearSVC']  # shape (77,)
assert pred_per_template.shape == (77,), pred_per_template.shape

# Lookup table: template_id -> predicted label. Vectorized fancy-index project.
max_tid = int(record_to_template.max())
tid_to_pred = np.full(max_tid + 1, fill_value=-1, dtype=np.int8)
for i, m in enumerate(id_mapping):
    tid_to_pred[m['template_id']] = int(pred_per_template[i])

pred_per_record = tid_to_pred[record_to_template]  # shape (74380,) int8
assert pred_per_record.shape == (74380,), pred_per_record.shape
assert (pred_per_record >= 0).all(), 'every record must have received a label (no -1 placeholders)'

# Expert ground-truth record labels (from nb03 §10).
record_to_binary = _load_numpy_local(CLUSTERS_DIR / 'record_to_binary_label.npy')
assert record_to_binary.shape == (74380,), record_to_binary.shape

n_total = int(pred_per_record.size)
pred_inf = int((pred_per_record == 1).sum())
pred_noise = int((pred_per_record == 0).sum())
exp_inf = int((record_to_binary == 1).sum())
exp_noise = int((record_to_binary == 0).sum())

assert pred_inf + pred_noise == n_total, 'predicted counts must sum to n_total'
assert exp_inf + exp_noise == n_total, 'expert counts must sum to n_total'

predicted_block = {
    'n_informative': pred_inf,
    'n_noise': pred_noise,
    'pct_informative': pred_inf / n_total,
    'pct_noise': pred_noise / n_total,
}
expert_block = {
    'n_informative': exp_inf,
    'n_noise': exp_noise,
    'pct_informative': exp_inf / n_total,
    'pct_noise': exp_noise / n_total,
}

side_by_side = pd.DataFrame.from_dict(
    {'predicted_LinearSVC': predicted_block, 'expert_groundtruth': expert_block},
    orient='index',
)[['n_informative', 'n_noise', 'pct_informative', 'pct_noise']]

with pd.option_context('display.float_format', '{:.4f}'.format):
    print(side_by_side)

print(
    f'\npct_informative — predicted: {100 * predicted_block["pct_informative"]:.2f} % | '
    f'expert: {100 * expert_block["pct_informative"]:.2f} %'
)

# --- Record-level confusion for LinearSVC ---

tp_rec = int(((record_to_binary == 1) & (pred_per_record == 1)).sum())
fp_rec = int(((record_to_binary == 0) & (pred_per_record == 1)).sum())
fn_rec = int(((record_to_binary == 1) & (pred_per_record == 0)).sum())
tn_rec = int(((record_to_binary == 0) & (pred_per_record == 0)).sum())
mcc_rec = float(matthews_corrcoef(record_to_binary, pred_per_record))

assert tp_rec + fp_rec + fn_rec + tn_rec == n_total

print(
    f'\nrecord-level LinearSVC: TP={tp_rec}, FP={fp_rec}, '
    f'FN={fn_rec}, TN={tn_rec}, MCC={mcc_rec:+.4f}'
)

filter_stats_payload = {
    'n_records_total': n_total,
    'predicted': predicted_block,
    'expert': expert_block,
    'record_confusion_linearsvc': {
        'TP': tp_rec, 'FP': fp_rec, 'FN': fn_rec, 'TN': tn_rec,
        'mcc': mcc_rec,
    },
}
filter_path = RESULTS_DIR / 'filter_stats_full.json'
save_json(filter_stats_payload, filter_path)
print(f'\nsaved: {filter_path}  ({filter_path.stat().st_size} bytes)')

loaded existing record_to_template: (74380,) int32
                     n_informative  n_noise  pct_informative  pct_noise
predicted_LinearSVC           3654    70726           0.0491     0.9509
expert_groundtruth            3648    70732           0.0490     0.9510

pct_informative — predicted: 4.91 % | expert: 4.90 %

record-level LinearSVC: TP=3618, FP=36, FN=30, TN=70696, MCC=+0.9905

saved: /Users/roman/Personal/dyploma/data/processed/classification/filter_stats_full.json  (460 bytes)


/Users/roman/Personal/dyploma/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Висновки

Три класифікатори ($\text{GaussianNB}$, $\text{LogReg}$, $\text{LinearSVC}$) натреновано на template-level парах $(\beta(t), \ell(t))$ з `sample_weight = support(t)`. Baseline-передиктор більшості показав $\text{MCC} = 0$ — це підтверджує, що будь-який нетривіальний результат вищих моделей є реальним сигналом, а не артефактом дисбалансу. Cross-check MCC проти rule-based міток ($\ell^{(0)}$) обчислено для майбутніх ітерацій з експертними коригуваннями (теза 2.3 Перевірка 2).

Додаткові артефакти для розділів 3.5–3.6 тези:

* `data/processed/classification/metrics_fb.csv` — основні метрики (MCC, F1-macro, ROC-AUC) на тестовому сплеті + повнодоменні MCC.
* `data/processed/classification/confusion_full.csv` — TP/FP/FN/TN та tied-back $\mathrm{MCC}_{\text{full,expert}}$ для всіх чотирьох моделей.
* `data/processed/classification/linearsvc_errors_full.json` — поіменні TP/FP/FN/TN списки для LinearSVC (фінального детектора).
* `data/processed/classification/filter_stats_full.json` — запис-рівневі статистики фільтра: передбачені vs експертні $n_{\text{інф}}, n_{\text{шум}}$, відсотки та запис-рівнева матриця плутанини.
* `data/processed/records/record_to_template.npy` — допоміжний кеш `(74380,) int32` для майбутніх запис-рівневих ноутбуків (зокрема 05).

Наступний крок — якісний аналіз помилок з attention-картами BERT (Перевірка 3 тези 2.3, ноутбук 05).
